In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Conv1D, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from pytorch_tabnet.tab_model import TabNetClassifier

# Load your dataset
df = pd.read_csv("Overview.csv")

# Create binary label based on sensor faults
df['Faulty Transformer'] = df[['WTI', 'OTI_A', 'OTI_T', 'MOG_A']].any(axis=1).map({True: 'Yes', False: 'No'})
df.drop(columns=['WTI', 'OTI_A', 'OTI_T', 'MOG_A'], inplace=True)

# Encode label: Yes = 1 (fault), No = 0 (healthy)
df['Faulty Transformer'] = df['Faulty Transformer'].map({'Yes': 1, 'No': 0})

# Separate features and labels
X = df.drop(columns=['Faulty Transformer'])
X = X.select_dtypes(include=[np.number])  # Drop non-numeric columns
y = df['Faulty Transformer']

# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Define models
models = {
    "SVM": SVC(kernel='rbf', probability=True),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "ANN": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
}

# Train models and collect predicted probabilities
predicted_probs = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted_probs[name] = model.predict_proba(X_test)[:, 1]

# Deep Learning Models
# Reshape data for deep learning models
X_train_dl = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_dl = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# LSTM
lstm_model = Sequential([
    LSTM(64, input_shape=(X_train_dl.shape[1], X_train_dl.shape[2]), activation='tanh'),
    Dense(1, activation='sigmoid')
])
lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.fit(X_train_dl, y_train, epochs=20, batch_size=32, verbose=1)
predicted_probs["LSTM"] = lstm_model.predict(X_test_dl).flatten()

# GRU
gru_model = Sequential([
    GRU(64, input_shape=(X_train_dl.shape[1], X_train_dl.shape[2]), activation='tanh'),
    Dense(1, activation='sigmoid')
])
gru_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
gru_model.fit(X_train_dl, y_train, epochs=20, batch_size=32, verbose=1)
predicted_probs["GRU"] = gru_model.predict(X_test_dl).flatten()

# 1D-CNN
cnn_model = Sequential([
    Conv1D(32, kernel_size=1, activation='relu', input_shape=(X_train_dl.shape[1], X_train_dl.shape[2])),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])
cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train_dl, y_train, epochs=20, batch_size=32, verbose=1)
predicted_probs["1D-CNN"] = cnn_model.predict(X_test_dl).flatten()

# TabNet
tabnet_model = TabNetClassifier(verbose=1, seed=42)
tabnet_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric=['accuracy'], max_epochs=50, patience=10, batch_size=256)
predicted_probs["TabNet"] = tabnet_model.predict_proba(X_test)[:, 1]

# Plot ROC and AUC for all models
plt.figure(figsize=(12, 8))
for model_name, y_pred_proba in predicted_probs.items():
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.2f})')

# Plot the diagonal line (random guessing)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guessing')

# Add plot details
plt.title('ROC Curves for All Models', fontsize=16, fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.legend(loc='lower right', fontsize=12)
plt.grid()
plt.tight_layout()
plt.show()